In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from gensim.models import KeyedVectors
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [7]:
# Load dataset
df = pd.read_csv('/content/drive/MyDrive/TA DIAZ/datasetdiaz.csv', usecols=['HS', 'Tweet'])
df = df.dropna(subset=['Tweet', 'HS'])  # Hilangkan NaN

X = np.array(df['Tweet'])
y = np.array(df['HS'])

In [8]:
# Load Word2Vec model
model_word2vec = KeyedVectors.load_word2vec_format('/content/drive/MyDrive/TA DIAZ/ws_model300.bin', binary=True)

In [9]:
# Fungsi untuk mengubah teks menjadi vektor Word2Vec
def get_w2v_features(w2v_model, sentence_group):
    feature_vec = np.zeros(w2v_model.vector_size, dtype="float32")  # Inisialisasi vektor nol
    num_words = 0
    for word in sentence_group.split():
        if word in w2v_model:
            feature_vec += w2v_model[word]  # Tambahkan vektor kata
            num_words += 1
    if num_words > 0:
        feature_vec /= num_words  # Rata-rata jika ada kata yang valid
    return feature_vec

In [10]:
# Konversi teks ke fitur Word2Vec
X_w2v = np.array([get_w2v_features(model_word2vec, text) for text in X])


In [11]:
# Standardisasi fitur Word2Vec
scaler = StandardScaler()
X_w2v_scaled = scaler.fit_transform(X_w2v)

In [12]:
# Fungsi untuk split train-test
def split_train_test(X, y, test_size):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=25)
    num_classes = np.max(y) + 1
    y_train = to_categorical(y_train, num_classes)
    y_test = to_categorical(y_test, num_classes)
    return X_train, X_test, y_train, y_test

In [13]:
# Fungsi CNN Model
def cnn_model(X, y, test_size):
    # Split dataset
    X_train, X_test, y_train, y_test = split_train_test(X, y, test_size)

    # Reshape data agar sesuai input CNN
    X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
    X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

    # Define CNN Model
    model = Sequential([
        Conv1D(32, 5, activation='relu', input_shape=(X_train.shape[1], 1), padding='same'),
        MaxPooling1D(5, padding='same'),
        Dropout(0.3),
        Flatten(),
        Dense(32, activation='relu'),
        Dense(2, activation='sigmoid')  # Sesuai dengan kode asli Anda
    ])
    # Compile model
    optimizer = Adam(learning_rate=0.0001)
    model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])

    # Early Stopping untuk mencegah overfitting
    early_stopping = EarlyStopping(patience=5, restore_best_weights=True)

    # Train model
    model.fit(X_train, y_train, epochs=35, batch_size=16, verbose=1, validation_data=(X_test, y_test), callbacks=[early_stopping])

    # Evaluasi model
    loss, accuracy = model.evaluate(X_test, y_test)
    print(f"Test Loss: {loss:.4f}")
    print(f"Test Accuracy: {accuracy:.4f}")

    # Prediksi pada data uji
    y_pred = model.predict(X_test)

    # Hitung metrik evaluasi
    accuracy = accuracy_score(y_test.argmax(axis=1), y_pred.argmax(axis=1))
    precision = precision_score(y_test.argmax(axis=1), y_pred.argmax(axis=1), average='macro')
    recall = recall_score(y_test.argmax(axis=1), y_pred.argmax(axis=1), average='macro')
    f1 = f1_score(y_test.argmax(axis=1), y_pred.argmax(axis=1), average='macro')
    class_repot = classification_report(y_test.argmax(axis=1), y_pred.argmax(axis=1))

    return accuracy, precision, recall, f1, class_repot

In [14]:
# Menjalankan eksperimen 5 kali seperti di kode asli
num_experiments = 5
accuracy_scores = []
precision_scores = []
recall_scores = []
f1_scores = []

for i in range(num_experiments):
    print(f"Experiment {i+1}")

    # Jalankan CNN dengan Word2Vec
    accuracy, precision, recall, f1, class_repot = cnn_model(X_w2v_scaled, y, 0.2)

    accuracy_scores.append(accuracy)
    precision_scores.append(precision)
    recall_scores.append(recall)
    f1_scores.append(f1)

    # Print hasil tiap eksperimen
    print(f"Evaluation Metrics - Experiment {i+1}:")
    print(f"Accuracy Score: {accuracy:.4f}")
    print(f"Precision Score: {precision:.4f}")
    print(f"Recall Score: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print()
    print("Classification Report:")
    print(class_repot)
    print('-------------------------------------------------------------------')
    print()


Experiment 1
Epoch 1/35


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


575/575 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.5765 - loss: 0.6834 - val_accuracy: 0.6649 - val_loss: 0.6366
Epoch 2/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.6590 - loss: 0.6254 - val_accuracy: 0.6758 - val_loss: 0.5992
Epoch 3/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.6827 - loss: 0.5975 - val_accuracy: 0.6880 - val_loss: 0.5819
Epoch 4/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.7013 - loss: 0.5787 - val_accuracy: 0.6971 - val_loss: 0.5730
Epoch 5/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.7140 - loss: 0.5652 - val_accuracy: 0.6880 - val_loss: 0.5826
Epoch 6/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.7216 - loss: 0.5587 - val_accuracy: 0.6980 - val_loss: 0.5682
Epoch 7/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.7126 - loss: 0.5626 - val_accuracy: 0.7158 - val_loss: 0.5549
Epoch 8/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - accuracy: 0.7308 - loss: 0.5406 - val_accuracy: 0.7124 - val_

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


575/575 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.5853 - loss: 0.6774 - val_accuracy: 0.6410 - val_loss: 0.6364
Epoch 2/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.6601 - loss: 0.6296 - val_accuracy: 0.6802 - val_loss: 0.6007
Epoch 3/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.6860 - loss: 0.5986 - val_accuracy: 0.6854 - val_loss: 0.5883
Epoch 4/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.7033 - loss: 0.5876 - val_accuracy: 0.6588 - val_loss: 0.6192
Epoch 5/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.7094 - loss: 0.5742 - val_accuracy: 0.6936 - val_loss: 0.5737
Epoch 6/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.7072 - loss: 0.5644 - val_accuracy: 0.7015 - val_loss: 0.5572
Epoch 7/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.7111 - loss: 0.5632 - val_accuracy: 0.7050 - val_loss: 0.5564
Epoch 8/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.7268 - loss: 0.5549 - val_accuracy: 0.6923 - val_

...

575/575 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.5849 - loss: 0.6813 - val_accuracy: 0.6527 - val_loss: 0.6484
Epoch 2/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.6390 - loss: 0.6418 - val_accuracy: 0.6619 - val_loss: 0.6209
Epoch 3/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.6857 - loss: 0.6067 - val_accuracy: 0.6849 - val_loss: 0.5942
Epoch 4/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.6932 - loss: 0.5898 - val_accuracy: 0.6932 - val_loss: 0.5805
Epoch 5/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.7026 - loss: 0.5781 - val_accuracy: 0.7010 - val_loss: 0.5726
Epoch 6/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.7142 - loss: 0.5569 - val_accuracy: 0.7019 - val_loss: 0.5659
Epoch 7/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.7063 - loss: 0.5674 - val_accuracy: 0.7054 - val_loss: 0.5598
Epoch 8/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.7180 - loss: 0.5546 - val_accuracy: 0.7115 - val_

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


575/575 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.5791 - loss: 0.6784 - val_accuracy: 0.6619 - val_loss: 0.6389
Epoch 2/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.6646 - loss: 0.6243 - val_accuracy: 0.6745 - val_loss: 0.5994
Epoch 3/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.6840 - loss: 0.5967 - val_accuracy: 0.6854 - val_loss: 0.5881
Epoch 4/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.7110 - loss: 0.5777 - val_accuracy: 0.6932 - val_loss: 0.5731
Epoch 5/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.6964 - loss: 0.5785 - val_accuracy: 0.7019 - val_loss: 0.5673
Epoch 6/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.7110 - loss: 0.5642 - val_accuracy: 0.7058 - val_loss: 0.5608
Epoch 7/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.7191 - loss: 0.5616 - val_accuracy: 0.7102 - val_loss: 0.5557
Epoch 8/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.7210 - loss: 0.5538 - val_accuracy: 0.7163 - val_

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


575/575 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.5695 - loss: 0.6795 - val_accuracy: 0.6205 - val_loss: 0.6454
Epoch 2/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.6538 - loss: 0.6338 - val_accuracy: 0.6688 - val_loss: 0.6091
Epoch 3/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.6791 - loss: 0.6065 - val_accuracy: 0.6984 - val_loss: 0.5850
Epoch 4/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.6956 - loss: 0.5855 - val_accuracy: 0.7058 - val_loss: 0.5736
Epoch 5/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.7083 - loss: 0.5714 - val_accuracy: 0.7102 - val_loss: 0.5658
Epoch 6/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.7165 - loss: 0.5614 - val_accuracy: 0.7154 - val_loss: 0.5596
Epoch 7/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.7148 - loss: 0.5575 - val_accuracy: 0.7150 - val_loss: 0.5548
Epoch 8/35
575/575 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.7266 - loss: 0.5510 - val_accuracy: 0.7180 - val_

In [15]:
# Rata-rata hasil dari 5 eksperimen
avg_accuracy = np.mean(accuracy_scores)
avg_precision = np.mean(precision_scores)
avg_recall = np.mean(recall_scores)
avg_f1 = np.mean(f1_scores)

print("\nAverage Evaluation Metrics:")
print(f"Average Accuracy Score: {avg_accuracy:.4f}")
print(f"Average Precision Score: {avg_precision:.4f}")
print(f"Average Recall Score: {avg_recall:.4f}")
print(f"Average F1 Score: {avg_f1:.4f}")



Average Evaluation Metrics:
Average Accuracy Score: 0.7577
Average Precision Score: 0.7565
Average Recall Score: 0.7520
Average F1 Score: 0.7533
